In [39]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
import torch

In [41]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 20, 5, 1)
        self.conv2 = nn.Conv2d(20, 50, 5, 1)
        self.fc1 = nn.Linear(4*4*50, 500)
        self.fc2 = nn.Linear(500, 10)
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2, 2)
        x = x.view(-1, 4*4*50)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

In [43]:
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('data', train=True, download=False,
                   transform=transforms.Compose([
                       transforms.ToTensor(),
                       transforms.Normalize((0.1307,), (0.3081,))])),
    batch_size=64, shuffle=True)

In [13]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

In [45]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Net().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.5)

In [47]:
def train(model, device, train_loader, optimizer, epoch):
		# 模型调整为训练模式，启用 batch normalization 和 dropout
    model.train()
    # 分batch从数据集中取数据
    for batch_idx, (data, target) in enumerate(train_loader):
		    # 解析一个batch中的数据
        data, target = data.to(device), target.to(device)
        # 梯度清零
        optimizer.zero_grad()
        # 数据输入模型
        output = model(data)
        # 计算损失函数
        loss = F.nll_loss(output, target)
        # 反向传播
        loss.backward()
        # 参数更新
        optimizer.step()
        # 打印信息
        if batch_idx % 100 == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(epoch, batch_idx * len(data), len(train_loader.dataset),
                       100. * batch_idx / len(train_loader), loss.item()))

In [49]:
for epoch in range(1, 4):
    train(model, device, train_loader, optimizer, epoch)

Train Epoch: 1 [0/60000 (0%)]	Loss: 2.318156
Train Epoch: 1 [6400/60000 (11%)]	Loss: 0.519885
Train Epoch: 1 [12800/60000 (21%)]	Loss: 0.190013
Train Epoch: 1 [19200/60000 (32%)]	Loss: 0.199725
Train Epoch: 1 [25600/60000 (43%)]	Loss: 0.201151
Train Epoch: 1 [32000/60000 (53%)]	Loss: 0.111248
Train Epoch: 1 [38400/60000 (64%)]	Loss: 0.158591
Train Epoch: 1 [44800/60000 (75%)]	Loss: 0.056514
Train Epoch: 1 [51200/60000 (85%)]	Loss: 0.038651
Train Epoch: 1 [57600/60000 (96%)]	Loss: 0.153273
Train Epoch: 2 [0/60000 (0%)]	Loss: 0.108243
Train Epoch: 2 [6400/60000 (11%)]	Loss: 0.121595
Train Epoch: 2 [12800/60000 (21%)]	Loss: 0.065562
Train Epoch: 2 [19200/60000 (32%)]	Loss: 0.132513
Train Epoch: 2 [25600/60000 (43%)]	Loss: 0.124594
Train Epoch: 2 [32000/60000 (53%)]	Loss: 0.025362
Train Epoch: 2 [38400/60000 (64%)]	Loss: 0.022427
Train Epoch: 2 [44800/60000 (75%)]	Loss: 0.215725
Train Epoch: 2 [51200/60000 (85%)]	Loss: 0.068458
Train Epoch: 2 [57600/60000 (96%)]	Loss: 0.098546
Train Epoch:

In [51]:
test_loader = torch.utils.data.DataLoader(
    datasets.MNIST('data', train=False, download=True, transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])),
    batch_size=1000, shuffle=True)

In [53]:
def test(model, device, test_loader):
		# 切换到验证模式，关闭 batch normalization 和 dropout
    model.eval()
    # 指标数值初始化
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            # 计算负对数似然损失
            test_loss += F.nll_loss(output, target, reduction='sum').item()
            # 获得类别预测
            pred = output.argmax(dim=1, keepdim=True)
            # 计算准确率
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(test_loader.dataset)
    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(test_loss, correct, len(test_loader.dataset),100. * correct / len(test_loader.dataset)))

In [55]:
test(model, device, test_loader)
torch.save(model, "model1.pth")


Test set: Average loss: 0.0504, Accuracy: 9856/10000 (99%)



In [57]:
total = sum([param.nelement() for param in model.parameters()])
# 精确地计算：1MB=1024KB=1048576字节
print('Number of parameter: % .7fM' % (total / 1e6))

Number of parameter:  0.4310800M


In [59]:
from copy import deepcopy
model_for_prune = deepcopy(model)

In [61]:
import torch.quantization
model_quantized = torch.quantization.quantize_dynamic(model_for_prune, {torch.nn.Linear}, dtype=torch.qint8)

In [63]:
test(model_quantized, device, test_loader)


Test set: Average loss: 0.0504, Accuracy: 9855/10000 (99%)



In [65]:
torch.save(model_quantized, "model2.pth")

In [67]:
total = sum([param.nelement() for param in model_quantized.parameters()])
# 精确地计算：1MB=1024KB=1048576字节
print('Number of parameter: % .7fM' % (total / 1e6))

Number of parameter:  0.0255700M
